# Aphelion HYDRA Colab Training

This notebook assumes:
- the repo stays code-only on GitHub
- the prepared dataset lives on Google Drive at `MyDrive/Aphelion/data/processed/XAUUSD/`
- you are running on a GPU runtime, ideally an A100

Required Drive files:
- `xauusd_hydra.parquet`
- `train.npz`
- `val.npz`
- `test.npz`
- `dataset_meta.json`
- `scaler.json`


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/MatinDeevv/Aphelion-Reasearch.git"
BRANCH = "main"
REPO_DIR = Path("/content/Aphelion-Reasearch")

DRIVE_ROOT = Path("/content/drive/MyDrive/Aphelion")
DATASET_DIR = DRIVE_ROOT / "data" / "processed" / "XAUUSD"
CHECKPOINT_DIR = DRIVE_ROOT / "models" / "hydra"
TINY_CHECKPOINT_DIR = DRIVE_ROOT / "models" / "hydra_tiny_validator_colab"

print(f"Repo URL: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")


In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=False)
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
if "A100" not in gpu_name.upper():
    print("Warning: this notebook will run on other GPUs, but the intended target is an A100.")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print("CUDA and TF32 are enabled.")


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

required_files = [
    "xauusd_hydra.parquet",
    "train.npz",
    "val.npz",
    "test.npz",
    "dataset_meta.json",
    "scaler.json",
]

missing = [name for name in required_files if not (DATASET_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing prepared dataset files on Drive: " + ", ".join(missing)
    )

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
TINY_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Drive mount is ready.")
print("Prepared dataset files found:")
for name in required_files:
    path = DATASET_DIR / name
    size_mb = path.stat().st_size / 1024**2
    print(f"  {name:20s} {size_mb:10.1f} MB")


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]", "loguru", "tqdm", "pyarrow"], check=True)

subprocess.run(["git", "rev-parse", "HEAD"], check=True)
print(f"Working directory: {Path.cwd()}")


In [ ]:
meta = json.loads((DATASET_DIR / "dataset_meta.json").read_text(encoding="utf-8"))
print(json.dumps(meta, indent=2)[:6000])


In [ ]:
tiny_cmd = [
    sys.executable,
    "scripts/train_hydra_tiny.py",
    "--data",
    str(DATASET_DIR),
    "--device",
    "cuda",
    "--epochs",
    "1",
    "--batch-size",
    "32",
    "--max-centers-per-split",
    "1024",
    "--checkpoint-dir",
    str(TINY_CHECKPOINT_DIR),
]

print("Running tiny validator:")
print(" ".join(tiny_cmd))
subprocess.run(tiny_cmd, check=True)


In [ ]:
full_cmd = [
    sys.executable,
    "scripts/train_hydra.py",
    "--data",
    str(DATASET_DIR),
    "--full",
    "--epochs",
    "20",
    "--batch-size",
    "32",
    "--checkpoint-dir",
    str(CHECKPOINT_DIR),
]

print("Launching full HYDRA training:")
print(" ".join(full_cmd))
subprocess.run(full_cmd, check=True)


In [ ]:
print("Tiny checkpoints:")
for path in sorted(TINY_CHECKPOINT_DIR.glob("*")):
    if path.is_file():
        print(f"  {path.name:40s} {path.stat().st_size / 1024**2:10.1f} MB")

print("\nFull checkpoints:")
for path in sorted(CHECKPOINT_DIR.glob("*")):
    if path.is_file():
        print(f"  {path.name:40s} {path.stat().st_size / 1024**2:10.1f} MB")
